# 05b. 최신 공시 등급 대조 (G 규칙, 판정기준서 v1.1 6장, D64, 2026-11-23)

- 대상: 추천 빈도 상위 30개 상품. 7회 이상 21개 전부 + 6회 동률 73개 중 9개(조회 당시 정렬 순서, 선택 기준 없음).
- 조회: 상품명만 사용 (`src/latest_grade.lookup`). 금융투자협회 전자공시는 펀드 기본정보에 투자위험등급 항목이 없어(2026-11-23 확인) 펀드 정보 사이트 FunETF(G2 3순위, 참고)의 값을 썼다.
- `RERUN_LOOKUP = False`이면 저장된 조회 결과만 읽는다 (웹 조회 없음).

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 60)
from src.load_cot import load_cot, load_products
from src.judge import judge
from src.latest_grade import lookup, g4_rejudge, LABEL
AS_OF, CHECKED = "2026-11-24", "2026-11-23"
RERUN_LOOKUP = False
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of=AS_OF)
recs = ctx["recs"]
top = recs.groupby("rec_name").agg(추천=("cot_id", "size"), CoT=("cot_id", "nunique"), 등록=("json_grade", "first"))           .sort_values("추천", ascending=False, kind="stable").head(30)
cache = ROOT / "data" / "llm_cache" / "latest_grade_lookup_20261123.csv"
if RERUN_LOOKUP or not cache.exists():
    L = lookup(top.index.tolist(), CHECKED); L.insert(1, "추천 수", top["추천"].values); L.insert(2, "데이터 등급", top["등록"].values)
    L.to_csv(cache, index=False, encoding="utf-8-sig")
L = pd.read_csv(cache, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
# 조회한 30개가 대상 목록이다. 7회 이상 추천 21개는 모두 포함, 나머지 9개는 6회 추천 동률 73개 중 조회 당시 정렬 순서로 들어갔다
cnt = recs.groupby("rec_name").size()
assert set(cnt[cnt >= 7].index) <= set(L["상품명"]) and (cnt.reindex(L["상품명"]) >= 6).all(), "조회 목록이 상위 빈도 기준과 다름"
print("7회 이상:", (cnt >= 7).sum(), "/ 6회 동률:", (cnt == 6).sum(), "중", (cnt.reindex(L["상품명"]) == 6).sum(), "개 포함")
print("조회 상태:", L["상태"].value_counts().to_dict())

## 1. 최신 대조표 (G1 두 시점, G3 클래스, G5 확인 불가)

In [ ]:
L["등급 변경"] = [("같음" if a == b else "변경") if pd.notna(a) and pd.notna(b) else "" for a, b in zip(L["데이터 등급"], L["최신 등급"])]
print(L[["상품명", "추천 수", "데이터 등급", "최신 등급", "등급 변경", "상태", "조회 클래스"]].to_string())

## 2. G4 재판정 (최신 등급으로 범위 다시 판정)

In [ ]:
g = g4_rejudge(recs, ctx["cots"], ctx["range"], L)
g["분할"] = g["cot_id"].map(ctx["cots"]["split"])
print(pd.crosstab(g["데이터 시점"], g["최신 시점"], margins=True))
tag = g[g["G4 태그"] != ""]
print("'최신 기준 부적합' 추천:", len(tag), "/ CoT:", tag["cot_id"].nunique(), "/ 분할:", tag.groupby("분할")["cot_id"].nunique().to_dict())
print("새로 범위 안이 된 추천 (데이터 시점 범위 밖 → 최신 범위 안):", ((g["데이터 시점"] == "범위 밖") & (g["최신 시점"] == "범위 안")).sum())
per = g.groupby("rec_name").agg(추천=("cot_id", "size"), 최신기준부적합=("G4 태그", lambda s: (s != "").sum()),
                                  최신범위밖=("최신 시점", lambda s: (s == "범위 밖").sum()), 데이터범위밖=("데이터 시점", lambda s: (s == "범위 밖").sum()))
L = L.merge(per, left_on="상품명", right_index=True, how="left")
print(per[per["최신기준부적합"] > 0].to_string())

## 3. 저장 `outputs/latest_grade_check.csv` (상품 단위, 원문 없음)

In [ ]:
L["사람 확인"] = [("G4 등급 변경" if c == "변경" else "") + (" | " if c == "변경" and s.startswith("사람") else "") + (s if s.startswith("사람") else "")
                 for c, s in zip(L["등급 변경"], L["상태"])]
L["최신 명칭"] = L["최신 등급"].map(lambda x: LABEL.get(int(x), "") if pd.notna(x) else "")
cols = ["상품명", "추천 수", "데이터 등급", "최신 등급", "최신 명칭", "등급 변경", "상태", "조회 이름", "조회 클래스", "출처 순위", "URL", "확인일",
        "추천", "데이터범위밖", "최신범위밖", "최신기준부적합", "사람 확인"]
L[cols].to_csv(ROOT / "outputs" / "latest_grade_check.csv", index=False, encoding="utf-8-sig")
g.to_csv(ROOT / "outputs" / "latest_grade_rejudge.csv", index=False, encoding="utf-8-sig")
print("저장 완료. 사람 확인 상품:", (L["사람 확인"] != "").sum())

In [ ]:
from src.report import update_scorecard
R = lambda k, t, v, a, u: {"지표": k, "Training": t, "Validation": v, "전체": a, "단위": u, "출처": "05b_최신등급대조", "근거 결정": "G1~G5, D64", "기준일": AS_OF}
rows = [R(f"조회 상태: {s}", "", "", int(n), "상품") for s, n in L["상태"].value_counts().items()]
tg = g["G4 태그"] != ""
rows += [R("등급 변경 상품", "", "", int((L["등급 변경"] == "변경").sum()), "상품"),
         R("조회 대상 상품 추천", int((g["분할"] == "Training").sum()), int((g["분할"] == "Validation").sum()), len(g), "추천"),
         R("최신 기준 부적합 추천 (G4)", int((tg & (g["분할"] == "Training")).sum()), int((tg & (g["분할"] == "Validation")).sum()), int(tg.sum()), "추천"),
         R("데이터 범위 밖 → 최신 범위 안 추천", "", "", int(((g["데이터 시점"] == "범위 밖") & (g["최신 시점"] == "범위 안")).sum()), "추천")]
update_scorecard(rows, "G 최신 공시 등급 대조")
print("scorecard 저장")

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과